In [57]:
import pandas as pd
import json

In [58]:
from pathlib import Path

pathlist = Path("logs").rglob('*.log')
for path in pathlist:
    print(path)

logs\Participant10_task1_2026-03-11_16-07-33.052435.log
logs\Participant1_task1_2026-03-11_15-54-33.659512.log
logs\Participant20_task1_2026-03-11_16-09-33.714144.log


In [59]:
from pathlib import Path

list_of_logs = []

pathlist = Path("logs").rglob('*.log')
for path in pathlist:
    uid = str(path).split("_")[0].split("\\")[1]
    with open(path) as f:
        data = f.read().splitlines()

    data_dicts = [json.loads(line) for line in data]
    logs = pd.DataFrame(data_dicts)
    logs["uid"] = uid
    list_of_logs.append(logs)

data = pd.concat(list_of_logs)
data["timestamp"]= pd.to_datetime(data["timestamp"])

In [60]:
data.to_csv("data/raw_logs_compiled.csv", index=False)
data

,type,timestamp,sessionID,uid,query,docid,rank,page,url,windowLocation,fromURL,toURL,clicked,fromPage,toPage
0,TaskStarted,2026-03-11 16:06:00.867000+00:00,24cce0de-96b4-41eb-bd43-110c3e25313a,Participant10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,idSubmitted,2026-03-11 16:06:00.878000+00:00,24cce0de-96b4-41eb-bd43-110c3e25313a,Participant10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,queryBoxFocused,2026-03-11 16:06:02.336000+00:00,24cce0de-96b4-41eb-bd43-110c3e25313a,Participant10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,querySubmitted,2026-03-11 16:06:08.298000+00:00,24cce0de-96b4-41eb-bd43-110c3e25313a,Participant10,can pigs climb trees,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,searchResultGenerated,2026-03-11 16:06:09.649000+00:00,24cce0de-96b4-41eb-bd43-110c3e25313a,Participant10,can pigs climb trees,4790,1,1,https://www.commonlit.org/texts/the-three-litt...,http://localhost:7001/result?query=can pigs cl...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,clickedResult,2026-03-11 16:09:23.281000+00:00,07403d1c-12db-450a-8e4f-c81f7d760330,Participant20,climate change effects,3035,1,6,https://www.commonlit.org/texts/water-scarcity...,https://www.commonlit.org/texts/water-scarcity...,NaN,NaN,NaN,NaN,NaN
103,cursorLeftSnippet,2026-03-11 16:09:23.668000+00:00,07403d1c-12db-450a-8e4f-c81f7d760330,Participant20,climate change effects,3035,1,6,https://www.commonlit.org/texts/water-scarcity...,http://localhost:7001/result?query=climate+cha...,NaN,NaN,NaN,NaN,NaN
104,wentBack,2026-03-11 16:09:28.196000+00:00,07403d1c-12db-450a-8e4f-c81f7d760330,Participant20,climate change effects,NaN,NaN,NaN,NaN,NaN,https://www.commonlit.org/texts/water-scarcity...,http://localhost:7001/result?query=climate+cha...,NaN,NaN,NaN
105,ClickedEndTask,2026-03-11 16:09:32.849000+00:00,07403d1c-12db-450a-8e4f-c81f7d760330,Participant20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
data.iloc[1]["timestamp"] - data.iloc[0]["timestamp"] 

Timedelta('0 days 00:00:00.011000')

In [62]:
results_data = data.loc[data["type"]=="searchResultGenerated"][["uid", "sessionID", "timestamp", "query", "docid", "url", "rank", "page"]]
results_data["retriever_rank"] = [int(rank) + (int(page)-1)*10 for (rank, page) in zip(results_data["rank"], results_data["page"])]
results_data = results_data.rename({"rank":"SERP_rank", "page": "SERP_page"}, axis=1)
results_data.drop_duplicates(inplace=True)

In [63]:
results_data.to_csv("data/results_data.csv", index=False)
results_data

,uid,sessionID,timestamp,query,docid,url,SERP_rank,SERP_page,retriever_rank
4,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:09.649000+00:00,can pigs climb trees,4790,https://www.commonlit.org/texts/the-three-litt...,1,1,1
5,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:09.649000+00:00,can pigs climb trees,5779,http://www.gutenberg.org/files/24940/24940-h/2...,2,1,2
6,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:09.649000+00:00,can pigs climb trees,7020,http://www.gutenberg.org/files/25359/25359-h/2...,3,1,3
7,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:09.650000+00:00,can pigs climb trees,3441,https://www.africanstorybook.org/#,4,1,4
8,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:09.650000+00:00,can pigs climb trees,2659,https://www.africanstorybook.org/,5,1,5
...,...,...,...,...,...,...,...,...,...
90,Participant20,07403d1c-12db-450a-8e4f-c81f7d760330,2026-03-11 16:09:16.498000+00:00,climate change effects,2436,https://en.wikipedia.org/wiki/Ultraviolet,6,6,56
91,Participant20,07403d1c-12db-450a-8e4f-c81f7d760330,2026-03-11 16:09:16.498000+00:00,climate change effects,4682,http://www.online-literature.com/william-dean-...,7,6,57
92,Participant20,07403d1c-12db-450a-8e4f-c81f7d760330,2026-03-11 16:09:16.499000+00:00,climate change effects,6298,http://www.gutenberg.org/files/3066/3066-h/306...,8,6,58
93,Participant20,07403d1c-12db-450a-8e4f-c81f7d760330,2026-03-11 16:09:16.499000+00:00,climate change effects,2452,https://en.wikipedia.org/wiki/Vacuum_energy,9,6,59


In [66]:
clicked_results_data = data.loc[data["type"]=="clickedResult"][["uid", "sessionID", "timestamp", "query", "docid", "url", "rank", "page"]]
clicked_results_data = clicked_results_data.rename({"page": "SERP_page", "rank": "SERP_rank"}, axis=1)
clicked_results_data.drop_duplicates(inplace=True)

In [67]:
clicked_results_data.to_csv("data/clicked_results_data.csv", index=False)
clicked_results_data

,uid,sessionID,timestamp,query,docid,url,SERP_rank,SERP_page
19,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:12.672000+00:00,can pigs climb trees,3092,https://www.africanstorybook.org/#,9,1
42,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:21.849000+00:00,can pigs climb trees,5567,http://www.gutenberg.org/files/28132/28132-h/2...,2,6
61,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:32.855000+00:00,can pigs climb trees,5463,http://www.gutenberg.org/files/28142/28142-h/2...,2,4
87,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:56.563000+00:00,what do pigs eat,5536,http://www.gutenberg.org/files/28130/28130-h/2...,3,1
93,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:07:05.790000+00:00,what do pigs eat,2513,https://www.africanstorybook.org/,10,1
114,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:07:15.988000+00:00,what do pigs eat,7034,http://www.gutenberg.org/files/25359/25359-h/2...,3,6
119,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:07:25.539000+00:00,what do pigs eat,4790,https://www.commonlit.org/texts/the-three-litt...,8,1
17,Participant1,64365d9f-f36b-47f6-a915-397b285677f6,2026-03-11 15:53:17.548000+00:00,can monkeys climb trees,3417,https://www.africanstorybook.org/,2,1
23,Participant1,64365d9f-f36b-47f6-a915-397b285677f6,2026-03-11 15:53:30.268000+00:00,can monkeys climb trees,3379,https://www.africanstorybook.org/,10,1
29,Participant1,64365d9f-f36b-47f6-a915-397b285677f6,2026-03-11 15:53:38.913000+00:00,can monkeys climb trees,2342,https://simple.wikipedia.org/wiki/Rainforest,6,1


In [68]:
session_data_rows = []

for uid in data["uid"].unique():
    user_sessions = data.loc[data["uid"]==uid]
    for session_id in user_sessions["sessionID"].unique():
        temp = user_sessions.loc[user_sessions["sessionID"]==session_id]
        start_time = temp.loc[temp["type"]=="TaskStarted"]["timestamp"].values[0]
        end_time = temp.loc[temp["type"]=="TaskEndConfirmed"]["timestamp"].values[0]
        session_data_rows.append([uid, session_id, start_time, end_time])

session_data = pd.DataFrame(session_data_rows, columns=["uid", "sessionID", "sessionStartTime", "sessionEndTime"])

In [69]:
session_data.to_csv("data/session_data.csv", index=False)
session_data

,uid,sessionID,sessionStartTime,sessionEndTime
0,Participant10,24cce0de-96b4-41eb-bd43-110c3e25313a,2026-03-11 16:06:00.867,2026-03-11 16:07:32.987
1,Participant1,64365d9f-f36b-47f6-a915-397b285677f6,2026-03-11 15:53:05.096,2026-03-11 15:54:33.592
2,Participant20,07403d1c-12db-450a-8e4f-c81f7d760330,2026-03-11 16:07:40.458,2026-03-11 16:09:33.650
